# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an end-to-end example for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
This dataset is published with a Croissant schema and is accessible at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed (use --upgrade for the latest version)
!pip install --upgrade mlcroissant

## 1. Data Loading
Load the Croissant metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, and for each, show the available field `@id`s and their Croissant data types.

In [ ]:
# List all record sets by @id and their associated field @ids and data types

record_sets = []
print("Available Record Sets:")
for rs in metadata.record_sets:
    print(f"- RecordSet @id: {rs.id}  | label: {rs.label}")
    record_sets.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    @id: {field.id:40} data_type: {field.data_type:10} name: {field.label}")
    print()

# Show the first record as a dictionary, for demonstration
if record_sets:
    demo_rs = record_sets[0]
    print(f"\nFirst record from record set @id='{demo_rs}':")
    for x in dataset.records(record_set=demo_rs):
        print(x)
        break

## 3. Data Extraction
Extract records from each record set into pandas DataFrames for analysis.

> Use the record set and field `@id`s from the previous section to select data dynamically.

In [ ]:
# Extract all records for all record sets as DataFrames (referenced by their @id)

dfs = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dfs[record_set_id] = pd.DataFrame(records)
    else:
        print(f"WARNING: No records found for RecordSet {record_set_id}!")

# For demonstration, show available DataFrames and their columns
for k, v in dfs.items():
    print(f"RecordSet @id: {k}  --> columns: {v.columns.tolist()}")
    display(v.head())

## 4. Exploratory Data Analysis (EDA)
Let's process demographic and mental health data, e.g. filter, normalize, and group on a numeric field such as `cr:field_gad7_total_score` and group by e.g. `cr:field_gender`. 

**Note:** All fields will be referenced by their exact Croissant `@id` as per best practices.

In [ ]:
# Pick main survey RecordSet (most likely one with GAD/PHQ/PSQ in fields)
# We'll try a common id like 'cr:recordSet_main' or print the available ones if unknown

# Pick first valid dataframe for demo
if dfs:
    # Heuristically pick record set with 'gad7' in any column
    main_rs = None
    for rs_id, df in dfs.items():
        lower_cols = [c.lower() for c in df.columns]
        if any('gad7' in col or 'phq9' in col or 'psq' in col for col in lower_cols):
            main_rs = rs_id
            break
    if not main_rs:
        # Default to first
        main_rs = list(dfs.keys())[0]

    df = dfs[main_rs]
    print(f"Using RecordSet @id={main_rs}")
    print(f"Columns: {df.columns.tolist()}")
else:
    print("No dataframes extracted.")

In [ ]:
# --- Reference all fields by their Croissant @id ---
# Example field ids (replace with actual ids if known):
#   Numeric: e.g. 'cr:field_gad7_total_score', Group: 'cr:field_gender'

numeric_field = None
group_field = None
# Find first numeric field
for field in metadata.get_record_set(main_rs).fields:
    if 'gad7' in field.id.lower() or (field.data_type and field.data_type.lower() in ('integer','float','number')):
        numeric_field = field.id
        print(f"Selected numeric_field: {numeric_field} ({field.label})")
        break
# Find group field
for field in metadata.get_record_set(main_rs).fields:
    if 'gender' in field.id.lower():
        group_field = field.id
        print(f"Selected group_field: {group_field} ({field.label})")
        break

if numeric_field not in df.columns:
    print(f"Warning: '{numeric_field}' field not found in DataFrame columns. Available: {df.columns.tolist()}")
if group_field not in df.columns:
    print(f"Warning: '{group_field}' field not found in DataFrame columns. Available: {df.columns.tolist()}")
# Continue EDA only if fields found
if numeric_field in df.columns:
    threshold = df[numeric_field].mean()
    # Filter records where score > mean (arbitrary demo threshold)
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Total records: {len(df)}; Filtered (>{threshold:.2f}): {len(filtered_df)}")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group by group_field
    if group_field and group_field in filtered_df.columns:
        grouped = (
            filtered_df.groupby(group_field)[numeric_field]
            .agg(['count','mean','std'])
            .sort_values('mean', ascending=False)
        )
        print(f"Mean {numeric_field} by {group_field}:")
        display(grouped)
else:
    print(f"Cannot proceed: No suitable numeric_field and group_field found.")

## 5. Visualization
Visualize the distribution of the selected numeric field (e.g. GAD-7 total score), and compare by gender if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=10, color='purple')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # Boxplot by group_field
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field], palette='pastel')
        plt.title(f"{numeric_field} Distribution by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No suitable numeric_field for visualization.")

## 6. Conclusion
- Successfully loaded and explored the FAIR² Mental Health dataset from Kilifi County, Kenya via its Croissant schema.
- Examined all record sets, fields, and extracted survey responses into pandas DataFrames.
- Demonstrated standard EDA and visualizations on GAD-7/mental health scores referenced by Croissant `@id`.
- The dataset is AI-ready for further modeling or statistical research. For additional context, refer to the dataset Croissant metadata or accompanying article.